In [1]:
import pandas as pd
import pathlib
from sklearn.preprocessing import StandardScaler
import joblib
import numpy as np

In [ ]:
# Path to ElasticNet models
model_path = pathlib.Path("../6.RNAseq/joblib").resolve()

latent_pred_file = collab_preds_dir / "phgg_latent_predictions.parquet"

In [ ]:
# Define the location of the saved models and output directory for results
output_dir = pathlib.Path("../6.RNAseq/results")
output_dir.mkdir(parents=True, exist_ok=True)

# File for combined correlation results
final_test_results_file = output_dir / "test_r2.parquet"
final_test_predictions_file = output_dir /  "test_preds.parquet"

In [ ]:
output_dir = pathlib.Path("../5.drug-dependency/results")
output_dir.mkdir(parents=True, exist_ok=True)
final_output_file = output_dir / "combined_latent_drug_correlations.parquet"

In [3]:
# Load new RNA-seq data
rna_path = pathlib.Path("./data").resolve()
rna_file = pathlib.Path(f"{rna_path}/shc_collaboration_rna_data.parquet").resolve()
train = pd.read_parquet("../6.RNAseq/data/RNASeq_train_zscored.parquet")

raw_rna_data = pd.read_parquet("../6.RNAseq/data/RNASeq.parquet")
raw_rna_data.rename(columns={'Unnamed: 0': 'SampleID'}, inplace=True)

new_rnaseq_data = pd.read_parquet(rna_file)
print("New RNA-seq data shape:", new_rnaseq_data.shape)

New RNA-seq data shape: (30, 1999)


In [4]:
new_rnaseq_data.head()

SampleID,TSPAN6 (7105),CFH (3075),FUCA2 (2519),SEMA3F (6405),TMEM176A (55365),KLHL13 (90293),ICA1 (3382),DBNDD1 (79007),TFPI (7035),SLC7A2 (6542),...,TXNIP (10628),RASSF5 (83593),SMIM22 (440335),FBXO17 (115290),MMP28 (79148),CCL5 (6352),SNURF (8926),HNF1B (6928),ARHGAP23 (57636),MARCKS (4082)
radiation,-0.160677,-0.521239,-1.080615,-0.794907,0.004587,0.393460,-1.084055,0.835352,-1.119984,0.341734,...,-0.325319,-0.244800,-0.644208,-0.193827,-0.658728,-1.297614,-1.868904,-0.656505,-0.414409,-0.075652
BT245_SHC202,-1.764925,-1.643278,-0.394011,-0.844701,-1.105973,-0.655252,0.228468,1.378580,-0.630617,0.137688,...,1.443568,-2.195903,-0.644208,0.549703,-1.375841,-1.297614,-1.748413,-0.656505,-0.762961,0.698580
DIPG4_SHC202,-0.007410,0.009551,-0.262572,1.825237,-1.105973,-1.329288,0.203261,-0.815949,1.824220,0.618901,...,0.983485,1.429113,-0.644208,0.448795,-0.303522,1.721217,-0.451200,-0.656505,0.218406,-1.506040
DIPG7_SHC202,-0.996181,0.352642,-0.617153,0.541690,-1.105973,1.402347,0.168341,0.941378,0.058862,1.011684,...,0.922983,1.884078,-0.644208,0.779673,-1.375841,-1.297614,-2.161789,-0.656505,-1.107240,-0.020135
DIPG13_SHC202,-2.003025,-0.435933,-1.811157,1.029807,-1.105973,-0.192524,0.378494,-0.852953,0.674367,0.969774,...,2.232004,0.364105,-0.644208,0.724473,-0.801213,1.226078,-0.838787,0.844166,0.781932,-0.217890


In [5]:
# Get the list of columns in the train data
train_columns = train.columns

# Filter the new RNAseq data to include only the columns in the train data
filtered_rnaseq_data = new_rnaseq_data.loc[:, train_columns.intersection(new_rnaseq_data.columns)]

# Add any missing columns with default values
for col in train_columns:
    if col not in filtered_rnaseq_data and col != "SampleID":
        filtered_rnaseq_data[col] = 0

In [ ]:
# Assuming shc_rnaseq_data is already a pandas dataframe
# Remove 'SampleID' column if it is not needed for the comparison
data = new_rnaseq_data.copy()

# Calculate the mean for each gene (column)
mean_values = data.mean()

# Calculate the Euclidean distance for each row from the mean values
new_rnaseq_data['Euclidean_Distance'] = np.linalg.norm(data - mean_values, axis=1)

SampleID: radiation, Euclidean Distance: 42.970144389756136
SampleID: BT245_SHC202, Euclidean Distance: 51.90117878951303
SampleID: DIPG4_SHC202, Euclidean Distance: 49.644402405856255
SampleID: DIPG7_SHC202, Euclidean Distance: 40.541390090632575
SampleID: DIPG13_SHC202, Euclidean Distance: 42.2578694564276
SampleID: GBM1_SHC202, Euclidean Distance: 42.09883852767042
SampleID: 245-1, Euclidean Distance: 44.48060086372874
SampleID: 245-2, Euclidean Distance: 45.217853178753316
SampleID: D4-1, Euclidean Distance: 51.999417015179446
SampleID: D4-2, Euclidean Distance: 52.42003029815361
SampleID: D7-1, Euclidean Distance: 46.4475873006523
SampleID: D7-2, Euclidean Distance: 44.88381670393222
SampleID: G1-1, Euclidean Distance: 49.584509644954174
SampleID: G1-2, Euclidean Distance: 50.55210563428106
SampleID: GBM2, Euclidean Distance: 50.4655983955974
SampleID: DIPG17, Euclidean Distance: 53.03160270456179
SampleID: GSM7305242, Euclidean Distance: 37.56897429747185
SampleID: GSM7305243, Eu

In [ ]:
# Scale data
scaler = StandardScaler()
rnaseq_data_scaled = scaler.fit_transform(filtered_rnaseq_data)

# Create an empty list to store DataFrames for each model
all_latent_dfs = []

# Iterate over all files in the saved_models directory
for model_file in model_path.glob("*.joblib"):
    # Extract model name and number of components from the filename
    model_file_name = model_file.stem
    try:
        parts = model_file_name.split("_")
        model_name = parts[1]  # First part is the model name
        dims = int(parts[3])
        init = int(parts[7])
        # Handle the case with multiple `z_` components by ensuring we grab the correct part
        z_value = int(parts[5])
    except (IndexError, ValueError):
        print(f"Skipping file {model_file} due to unexpected filename format.")
        continue

    # Load the model
    print(f"Loading model from {model_file}")
    
    model = joblib.load(model_file)
    # Predict latent variables using the best model
    latent_predictions = model.predict(rnaseq_data_scaled)

    latent_df = pd.DataFrame(latent_predictions, columns=["latent_score"])

    # Add the sample names (row indices) as a new column
    latent_df["ModelID"] = new_rnaseq_data.index

    # Add model name and z_value columns
    latent_df["model"] = model_name
    latent_df["z"] = z_value
    latent_df["latent_dim_total"] = dims
    latent_df["init"] = init

    # Append to the list of DataFrames
    all_latent_dfs.append(latent_df)

# Combine all DataFrames into one
final_latent_df = pd.concat(all_latent_dfs, ignore_index=True)
print("Latent DataFrame shape:", final_latent_df.shape)

collab_preds_dir = pathlib.Path("../7.collab-data/results").resolve()
collab_preds_dir.mkdir(parents=True, exist_ok=True)

final_latent_df.to_parquet(latent_pred_file)

Loading model from /home/juliacurd/gene_dependency_representations_real/6.RNAseq/joblib/elasticnet_vanillavae_dims_200_z_41_init_0_seed_1547276024.joblib
[0.23329945 0.23738222 0.18839211 0.28441832 0.22785571 0.25042196
 0.21234919 0.21525133 0.1647303  0.16914666 0.19179012 0.21011979
 0.22391035 0.22536673 0.25343976 0.21634521 0.17833213 0.2348475
 0.17287108 0.18222752 0.17929533 0.14586128 0.17281985 0.11927155
 0.16621492 0.13878131 0.2023905  0.16959693 0.16235658 0.14030189]
Loading model from /home/juliacurd/gene_dependency_representations_real/6.RNAseq/joblib/elasticnet_vanillavae_dims_150_z_40_init_0_seed_1547276024.joblib
[0.17909923 0.25882445 0.10756785 0.23469417 0.08341112 0.32058349
 0.21377625 0.19644523 0.15244833 0.17212676 0.27885252 0.26631869
 0.37423672 0.38033676 0.34801781 0.43548924 0.13878418 0.10601835
 0.11356923 0.15444728 0.10757172 0.08326206 0.17310695 0.0799136
 0.04174831 0.16337987 0.07362246 0.07783418 0.04654225 0.03652135]
Loading model from /ho

/home/juliacurd/anaconda3/envs/gene_dependency_representations_real/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/juliacurd/anaconda3/envs/gene_dependency_representations_real/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/juliacurd/anaconda3/envs/gene_dependency_representations_real/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [8]:
final_latent_df.head(50)

,latent_score,ModelID,model,z,latent_dim_total,init
0,0.233299,radiation,vanillavae,41,200,0
1,0.237382,BT245_SHC202,vanillavae,41,200,0
2,0.188392,DIPG4_SHC202,vanillavae,41,200,0
3,0.284418,DIPG7_SHC202,vanillavae,41,200,0
4,0.227856,DIPG13_SHC202,vanillavae,41,200,0
5,0.250422,GBM1_SHC202,vanillavae,41,200,0
6,0.212349,245-1,vanillavae,41,200,0
7,0.215251,245-2,vanillavae,41,200,0
8,0.164730,D4-1,vanillavae,41,200,0
9,0.169147,D4-2,vanillavae,41,200,0


In [10]:
final_test_results_df = pd.read_parquet(final_test_results_file)
final_test_predictions_df = pd.read_parquet(final_test_predictions_file)

In [ ]:
combined_results_df = pd.read_parquet(final_output_file)

           z  full_model_z      model  init  \
0          0             7  betatcvae     2   
1          0             7  betatcvae     2   
2          0             7  betatcvae     2   
3          0             7  betatcvae     2   
4          0             7  betatcvae     2   
...       ..           ...        ...   ...   
158531521  1             2        nmf     0   
158531522  1             2        nmf     0   
158531523  1             2        nmf     0   
158531524  1             2        nmf     0   
158531525  1             2        nmf     0   

                                                  drug  pearson_correlation  \
0                     BRD-A00077618-236-07-6::2.5::HTS            -0.028614   
1                     BRD-A00100033-001-08-9::2.5::HTS            -0.028150   
2                     BRD-A00147595-001-01-5::2.5::HTS             0.023167   
3                     BRD-A00218260-001-03-4::2.5::HTS             0.034617   
4                     BRD-A00376169-001-

In [12]:
final_latent_df.sort_values(by='latent_score', ascending=False).head(50)

,latent_score,ModelID,model,z,latent_dim_total,init
75,0.789451,DIPG17,betatcvae,4,5,4
81,0.749647,GSM7305249,betatcvae,4,5,4
73,0.748670,G1-2,betatcvae,4,5,4
65,0.748516,GBM1_SHC202,betatcvae,4,5,4
72,0.740043,G1-1,betatcvae,4,5,4
74,0.738560,GBM2,betatcvae,4,5,4
61,0.737716,BT245_SHC202,betatcvae,4,5,4
79,0.732305,GSM7305246,betatcvae,4,5,4
85,0.732298,GSM7305253,betatcvae,4,5,4
70,0.728977,D7-1,betatcvae,4,5,4


In [ ]:
# Assuming filtered_df is the DataFrame
# Group by 'model', 'z', and 'latent_dim_total' and calculate the variation in 'Latent_0'
variation_df = (
    final_latent_df.groupby(['model', 'z', 'latent_dim_total', 'init'])
    .agg(
        max_latent_score=('latent_score', 'max'),
        min_latent_score=('latent_score', 'min'),
        std_latent_score=('latent_score', 'std'),
    )
    .reset_index()
)

# Add a column for the range of Latent_0
variation_df['range_latent_score'] = variation_df['max_latent_score'] - variation_df['min_latent_score']

# Sort by range_latent_0 or std_latent_0 to find the groups with the biggest variation
sorted_variation_df = variation_df.sort_values(by='range_latent_score', ascending=False)

# Display the top groups
sorted_variation_df.head(50)

#This code describes the variation for scores, checking to see if top range is v low so no interesting results are expected b/w samples

,model,z,latent_dim_total,init,max_latent_score,min_latent_score,std_latent_score,range_latent_score
1,vanillavae,40,150,0,0.435489,0.036521,0.109987,0.398968
2,vanillavae,41,200,0,0.284418,0.119272,0.038844,0.165147
0,betatcvae,4,5,4,0.789451,0.675250,0.023787,0.114201
